# Galerie de réentraînement — Cas immobilier ImmoVista

> **Nouveauté importante** : ici on fait de la **régression**, pas de la
> classification. La cible est continue (un prix, pas une classe).
> C'est l'autre grande famille de problèmes en ML supervisé.

---

## 🎬 Situation

**Lundi 10h chez FastIA.** Karim te confie un nouveau dossier :

> **De :** Mathieu Lavergne, Directeur Produit — ImmoVista (estimation immobilière en ligne)
> **Objet :** Prototype de modèle d'estimation
>
> Bonjour FastIA,
>
> Nous lançons un service d'**estimation automatique du prix médian des
> logements** à l'échelle d'un quartier. Avant d'investir dans une
> collecte de données française à grande échelle, nous voulons **valider
> la démarche** sur un dataset public bien documenté (zones de
> recensement californiennes) — la transposition à notre marché viendra
> dans un second temps.
>
> Pouvez-vous nous entraîner un modèle de régression qui prédit le prix
> médian des logements à partir des caractéristiques socio-géographiques
> de la zone, en comparant 2 configurations d'hyperparamètres ?
>
> Cordialement,
> Mathieu Lavergne.

---

## 🧭 Le pattern, version régression

C'est le **même pattern en 6 étapes** que M1-B1, avec 3 différences :

| Étape | En classification (M1-B1) | En régression (ce notebook) |
|---|---|---|
| [3] Split | `stratify=y` | **Pas de stratify** (cible continue) |
| [4] Modèle | `RandomForestClassifier` | `RandomForestRegressor` |
| [5] Métriques | F1, ROC-AUC, matrice confusion | **MAE, RMSE, R²** |

Le reste (charger, EDA, persister) ne bouge pas.

## Setup

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## [1] Charger le dataset

Source : **California Housing** — fourni par scikit-learn, téléchargé une
seule fois lors du premier appel puis mis en cache local. Connexion
internet nécessaire au premier appel uniquement.

20 640 zones de recensement californiennes (1990), 8 indicateurs
socio-géographiques par zone.

Variables :
- `MedInc` : revenu médian de la zone (en dizaines de milliers $)
- `HouseAge` : âge médian des logements (années)
- `AveRooms` : nombre moyen de pièces par logement
- `AveBedrms` : nombre moyen de chambres par logement
- `Population` : population de la zone
- `AveOccup` : nombre moyen d'occupants par logement
- `Latitude` / `Longitude` : coordonnées géographiques

**Cible** (`MedHouseVal`) : prix médian des logements (en centaines de
milliers de $).

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
df = df.rename(columns={"MedHouseVal": "target"})

print(f"Lignes : {df.shape[0]} | Colonnes : {df.shape[1]}")
df.head()

## [2] Explorer (EDA mini)

En régression, on regarde plutôt :
1. **Distribution de la cible** — est-elle skewée ? bornée ? tronquée ?
2. **Présence de NaN** — pareil qu'en classification.
3. **Quelques relations clés** — variables les plus corrélées à la cible.

In [ ]:
print(f"Valeurs manquantes au total : {df.isna().sum().sum()}")

print("\nStatistiques de la cible (prix médian, en centaines de milliers $) :")
print(df["target"].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(data=df, x="target", bins=50, ax=axes[0])
axes[0].set_title("Distribution de la cible (prix médian)")
axes[0].set_xlabel("Prix (centaines de milliers $)")

sns.scatterplot(
    data=df.sample(2000, random_state=RANDOM_STATE),
    x="MedInc",
    y="target",
    alpha=0.3,
    ax=axes[1],
)
axes[1].set_title("Prix vs revenu médian de la zone")

sns.scatterplot(
    data=df.sample(2000, random_state=RANDOM_STATE),
    x="Longitude",
    y="Latitude",
    hue="target",
    palette="viridis",
    alpha=0.5,
    s=10,
    ax=axes[2],
)
axes[2].set_title("Carte des prix (couleur = cible)")

plt.tight_layout()
plt.show()

**Lecture rapide** : la cible est plafonnée à 5 (les valeurs > 500 000 $
ont été tronquées dans le dataset d'origine) — c'est un piège classique
à connaître, ça crée une barre verticale dans le scatter réel vs prédit.
Le revenu médian est très fortement corrélé au prix. La carte révèle
les deux grandes zones chères : baie de San Francisco au nord, agglo
Los Angeles / San Diego au sud.

## [3] Découper train / test

**Différence clé avec la classification** : la cible est continue, donc
**pas de `stratify`**. On garde `random_state=42` pour la reproductibilité.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Moyenne cible train : {y_train.mean():.2f}")
print(f"Moyenne cible test  : {y_test.mean():.2f}")

## [4] Entraîner — un plancher + 2 configurations

On se donne d'abord un **plancher** : un `DummyRegressor` qui prédit toujours
la moyenne du prix. C'est le minimum à battre — si un vrai modèle ne fait pas
mieux que « prédire la moyenne », il n'apprend rien.

Puis deux RandomForest :

- **Config A** : valeurs par défaut — référence.
- **Config B** : `n_estimators=300`, `max_depth=15`, `min_samples_leaf=2` —
  plus de profondeur pour capter les non-linéarités géographiques.

> ⚠️ **Pas de `StandardScaler`.** Toutes les features sont numériques, mais
> surtout un RandomForest est **insensible à l'échelle** (il découpe par
> seuils). Le scaler serait utile pour un modèle **linéaire**, pas ici.

In [ ]:
# Plancher : prédit toujours la moyenne de la cible.
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

# Pas de scaler : un RandomForest est insensible à l'échelle des features.
reg_a = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
reg_b = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=2,
)

reg_a.fit(X_train, y_train)
reg_b.fit(X_train, y_train)
print("Plancher + deux modèles entraînés.")

## [5] Évaluer sur le test

Métriques de régression :
- **MAE** (Mean Absolute Error) : erreur moyenne, en centaines de
  milliers $ ici — interprétable directement par le métier.
- **RMSE** (Root Mean Squared Error) : pénalise plus les grosses erreurs.
- **R²** : part de variance expliquée — entre 0 et 1 (idéalement).

In [ ]:
def evaluate(name: str, model: object) -> dict:
    y_pred = model.predict(X_test)
    return {
        "modèle": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred),
    }


scores = pd.DataFrame(
    [
        evaluate("Plancher — Dummy", dummy),
        evaluate("Config A — défaut", reg_a),
        evaluate("Config B — tuned", reg_b),
    ]
)
scores.round(3)

In [ ]:
best_row = scores[scores["modèle"].str.startswith("Config")]
best_name = best_row.loc[best_row["MAE"].idxmin(), "modèle"]
best_model = reg_b if "B" in best_name else reg_a
y_pred_best = best_model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.3)
axes[0].plot([0, y_test.max()], [0, y_test.max()], "r--", label="y = x")
axes[0].set_xlabel("Valeur réelle")
axes[0].set_ylabel("Prédiction")
axes[0].set_title(f"Réel vs prédit — {best_name}")
axes[0].legend()

residus = y_test - y_pred_best
sns.histplot(residus, bins=50, ax=axes[1])
axes[1].axvline(0, color="r", linestyle="--")
axes[1].set_title("Distribution des résidus (réel − prédit)")
axes[1].set_xlabel("Erreur")

plt.tight_layout()
plt.show()

## 🤔 Question réflexive — ces features seront-elles là en production ?

Le modèle s'appuie sur `MedInc`, `AveOccup`, `Population`… des **agrégats de
recensement** par zone. Le jour où ImmoVista veut estimer un bien précis en
ligne : ces agrégats sont-ils **disponibles**, **à jour**, et au bon niveau
géographique au moment de la prédiction ?

Une feature qui n'existe pas (ou pas encore) quand on prédit rend le modèle
inutilisable en l'état — **même avec un excellent R²**. C'est une cause très
fréquente de modèle qui « marche en notebook » mais pas en prod. Lesquelles de
ces 8 variables te semblent les plus risquées de ce point de vue ?

## [6] Persister le modèle

In [ ]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "immovista_estimation_v1.joblib"
joblib.dump(best_model, model_path, compress=3)

print(f"Modèle sauvé : {model_path}")
print(f"Taille : {model_path.stat().st_size / 1024:.1f} Ko")

## 📝 Verdict — à remettre à Mathieu Lavergne

> Sur ~4 100 zones de test (20 % du dataset), le modèle retenu (la meilleure
> des 2 configs, cf. tableau ci-dessus) atteint une MAE de l'ordre de 0.33
> (≈ 33 000 $) et un R² > 0.80. Le revenu médian et la position géographique
> expliquent la majeure partie du prix — ce qui est cohérent métier.

> **Limites** : le plafonnement de la cible à 500 000 $ tronque les
> zones très haut de gamme et fausse les prédictions sur ces cas (visible
> sur la barre verticale du scatter réel vs prédit). Avant tout usage
> sur le marché français, il faudra **recollecter les données** (les
> features californiennes des années 90 ne transposent pas) et **gérer
> explicitement les zones premium** sans plafonner.

---

## 🔎 Ce que tu viens de voir d'inédit

1. **Pas de `stratify`** dans le split — c'est une particularité de la
   classification, pas de la régression.
2. **Pas de `predict_proba`** — la régression ne donne pas de probabilité,
   juste une valeur.
3. **Métriques différentes** : MAE/RMSE/R² remplacent F1/ROC-AUC. Mais
   l'esprit est le même — on évalue sur le test, on compare les configs,
   on tranche.
4. **Visualisation différente** : pas de matrice de confusion (il n'y a
   pas de classes), mais un nuage `réel vs prédit` et une distribution
   des résidus.
5. **Cible plafonnée** : un piège classique des datasets immobiliers — à
   repérer en EDA avant de blâmer le modèle de mal prédire les zones
   chères.

## ⭐ Pour aller plus loin (optionnel)

- Remplace `RandomForestRegressor` par `GradientBoostingRegressor` —
  attention au temps d'entraînement.
- Essaie de prédire `log(target)` au lieu de `target` (cible skewée) et
  observe l'effet sur les résidus.
- Filtre les zones où `target == 5` (plafonnées) avant l'entraînement et
  regarde si la qualité s'améliore sur les zones non plafonnées.
- Affiche `best_model.feature_importances_` —
  quelles features dominent ?

---

*Galerie de réentraînement — cas immobilier ImmoVista.*